# Chapter 10 — Rough, Temporal, and Fuzzy Modelling
### Notebook 0 · Overview and setup

*Book reference: Keet, *Ontology Engineering* (2nd ed.), Ch. 10*

Three kinds of imperfection a crisp ontology cannot express — things that happen **over time**, predicates with **no sharp boundary**, and data too **coarse** to separate the cases you care about — and the three formalisms that handle them.

In [1]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


In [2]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch10_toolkit as ch10
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

In [3]:
import oe_course; print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


## Notebooks in this chapter

| # | Notebook | Book section | What you build |
|---|---|---|---|
| 0 | `00_overview_and_setup` | — | environment check |
| 1 | `01_temporal` | 10.1 | **Allen's interval algebra**, composition table derived |
| 2 | `02_vagueness_and_granularity` | 10.2 | fuzzy membership + rough approximations |
| 3 | `03_exercises` | 10.3 | autograded answers |
| 4 | `04_agentic_lab` | — | a formalism-choosing agent + a **propagation MDP** |


**By the end of this notebook you can:**

1. Use the thirteen Allen relations, and **derive** the composition table rather than trusting it.
2. Detect an inconsistent temporal network, and say why the check is sound but incomplete.
3. Model a vague predicate with fuzzy membership, and say what an alpha-cut costs you.
4. Report a set you cannot describe exactly as a lower and an upper approximation.
5. Choose between the three, and **price** the choice.

## The one idea shared by all three

> **Expressivity is never free.**

Each formalism buys the ability to say something a crisp ontology cannot, and each charges for it. The bill is not rhetorical: deciding consistency of a general Allen network is **NP-complete**, while fuzzy membership and rough approximation stay polynomial. The agent in Notebook 4 is scored on getting *both* the choice and its price right.

### Sanity check: all thirteen relations, and a derived composition table

In [4]:
intervals = [(s, e) for s in range(5) for e in range(s + 1, 6)]
observed = {ch10.relation_between(a, b) for a in intervals for b in intervals}
print(f'{len(observed)} of {len(ch10.ALLEN_RELATIONS)} relations observed')
table = ch10.composition_table()
print(f'{len(table)} composition entries derived (13 x 13 = 169)')
assert len(observed) == 13 and len(table) == 169

13 of 13 relations observed
169 composition entries derived (13 x 13 = 169)
